<a href="https://colab.research.google.com/github/ELIXIREstonia/2026-05-25-Python/blob/main/02_tabular_data_with_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pandas for Data Analysis

Welcome to the pandas workbook for Day 2. Pandas is the main Python library we will use for table-shaped data: reading files, checking data quality, selecting rows and columns, creating new variables, summarising groups, and preparing data for visualisation and statistics.

This notebook is written for guided teaching and for self-study. Read the explanations, run each code cell, change small pieces of code, and use the practice cells to check that you can do the same operation without copying the example exactly.


## Learning goals

By the end of this notebook, you should be able to:

- explain the difference between a `Series` and a `DataFrame`
- load a CSV file into pandas and inspect its structure
- select columns, filter rows, and sort data
- create new columns from existing columns
- handle basic missing values carefully
- combine small tables with `merge` and `concat`
- summarise data with `groupby`, `agg`, and `pivot_table`
- reshape data between wide and long formats
- prepare tidy data for later visualisation and a follow-up statistics course


Official pandas documentation is versioned. If you search online, check that the documentation version reasonably matches the pandas version you are using. This matters because methods, parameters, default behaviour, and examples can change between pandas versions. You can check your installed version with `pd.__version__`.


## How to use this notebook

Run the notebook from top to bottom the first time. After that, come back to the practice cells and try to solve them from memory.

A good learning rhythm is:

1. Run the example.
2. Predict what will change if you edit one value or column name.
3. Make the edit and run the cell again.
4. Explain the result in one sentence.


## Setup and data

We use a small example table first, then a real teaching dataset called `Islander_data`. The data file is included in this repository under `data/Islander_data.csv`. If you open the notebook in Colab without cloning the repository, the code falls back to the public CSV URL.


In [ ]:
# pathlib helps us work with file paths in a way that works across operating systems.
from pathlib import Path

# pandas is the main library for table-shaped data. The conventional alias is pd.
import pandas as pd
from IPython.display import display

# Show more columns and round displayed decimal values for easier reading.
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)

# Prefer the local course data file, but keep an online fallback for Colab.
DATA_PATH = Path("data/Islander_data.csv")
DATA_URL = "https://raw.githubusercontent.com/ELIXIREstonia/2026-05-25-Python/main/data/Islander_data.csv"


def load_islander_data():
    """Load the local course dataset, or use the online copy in Colab."""
    source = DATA_PATH if DATA_PATH.exists() else DATA_URL
    return pd.read_csv(source)

# Keep the original loaded data unchanged so we can return to it if needed.
islanders_raw = load_islander_data()
islanders_raw.head()

## About the dataset

`Islander_data` contains a small teaching dataset about a memory test in virtual participants. Each row is one participant. The main columns are:

- `first_name`, `last_name`: participant names
- `age`: participant age
- `Happy_Sad_group`: whether the participant was primed with happy (`H`) or sad (`S`) memories
- `Drug`: treatment code: `A`, `T`, or `S`
- `Dosage`: dose level from 1 to 3
- `Mem_Score_Before`, `Mem_Score_After`: memory scores before and after treatment
- `Diff`: after score minus before score

Use this as a learning dataset, not as medical evidence. The dataset is distributed under CC BY 4.0; see the repository `data/README.md` for attribution.


# 1. Series and DataFrames

A pandas `Series` is one labelled column. A `DataFrame` is a table made of Series objects that share the same row index.

You can think of a DataFrame as similar to a spreadsheet, but with Python methods for repeatable analysis.

The official reference pages are useful when you want to explore what each structure can do:

- [`Series` reference](https://pandas.pydata.org/docs/reference/series.html): methods and attributes for one column of data
- [`DataFrame` reference](https://pandas.pydata.org/docs/reference/frame.html): methods and attributes for a whole table

You do not need to memorise these pages. Use them like a dictionary: search for a method name you have seen, or browse method names when you need to check whether pandas already has a tool for the task.


In [ ]:
# A Series is one labelled column of data.
numbers = pd.Series([10, 20, 30, 40], name="score")
display(numbers)

# A DataFrame is a table: each key becomes a column name.
people = pd.DataFrame({
    "name": ["Anna", "Jan", "Maria", "Peter", "Liis"],
    "age": [22, 35, 19, 42, 28],
    "city": ["Tallinn", "Tartu", "Tallinn", "Parnu", "Tartu"],
    "income": [1200, 2100, 950, 1800, 1600],
})
people

Useful first questions for any DataFrame:

- How many rows and columns are there?
- What are the column names?
- What does one row represent?
- Which columns are numeric, categorical, text, or dates?


In [ ]:
# .shape returns (number_of_rows, number_of_columns).
print("Rows, columns:", people.shape)

# .columns stores the column names. list(...) makes them easier to read here.
print("Column names:", list(people.columns))

# The index labels the rows. Here it is the default numeric index.
print("Index:", people.index)

# .info() summarises column names, non-missing values, and data types.
people.info()

# 2. Data types

Each column has a data type. Data types matter because they control which operations make sense.

Common pandas types include:

- `object` or `string`: text
- `int64`, `Int16`, `Int32`, `Int64`: whole numbers
- `float64`: decimal numbers
- `bool`: `True` or `False`
- `datetime64`: dates and times
- `category`: repeated labels such as treatment groups

Pandas nullable integer types such as `Int16` and `Int32` are useful when a whole-number column may contain missing values.


In [ ]:
# Work on a copy so the original people DataFrame stays unchanged.
typed_people = people.copy()

# Use a nullable integer dtype for whole numbers that may later contain missing values.
typed_people["age"] = typed_people["age"].astype("Int16")

# category is useful for repeated labels such as city names.
typed_people["city"] = typed_people["city"].astype("category")

# This creates a Boolean column: True when the condition is met, False otherwise.
typed_people["is_adult"] = typed_people["age"] >= 18

# Convert text dates into pandas datetime values.
typed_people["joined"] = pd.to_datetime(["2024-01-10", "2024-02-12", "2024-02-20", "2024-03-05", "2024-03-18"])

display(typed_people)
typed_people.dtypes

## Practice: Inspect data types

Use the `islanders_raw` DataFrame.

1. Display the first 8 rows.
2. Print the number of rows and columns.
3. Use `.info()` to inspect data types.
4. Which columns would you treat as categories?


In [ ]:
# Write your solution here.
# Tip: try .head(), .shape, and .info().


# 3. Indexing and selecting data

Every DataFrame has an index. By default, it is a row number starting at 0. Sometimes it is useful to set a meaningful index such as a participant ID, a name, or a date.

Use `.loc[row_selection, column_selection]` when you select by labels or conditions. This is one of the most important pandas habits to build.


Two common indexers are `.loc[]` and `.iloc[]`:

- `.loc[]` selects rows and columns by labels or Boolean conditions.
- `.iloc[]` selects rows and columns by integer position.

Use `.loc[]` for most analysis filters. Use `.iloc[]` when you specifically mean "the first rows", "the last rows", or positions by number.


In [ ]:
# set_index() uses an existing column as row labels.
people_by_name = people.set_index("name")
display(people_by_name)

# .loc[] selects by index label. Here it returns Anna's row.
display(people_by_name.loc["Anna"])

# reset_index() turns the index back into a normal column.
people_by_name.reset_index()

In [ ]:
# Select columns by passing a list of column names.
display(people[["name", "age"]])

# .loc[row_condition, column_list] selects rows and columns in one expression.
adults = people.loc[people["age"] >= 25, ["name", "city", "income"]]
adults

## Filtering with conditions

A condition such as `people["age"] >= 25` creates a Boolean Series. Pandas keeps rows where the condition is `True`.

Combine conditions with:

- `&` for AND
- `|` for OR
- `~` for NOT

Put each condition in parentheses.


In [ ]:
# Build a filter with two conditions: city is Tartu OR income is at least 1800.
tartu_or_high_income = people.loc[
    (people["city"] == "Tartu") | (people["income"] >= 1800),
    ["name", "city", "income"],
]
tartu_or_high_income

## Practice: Selecting and filtering

Using `islanders_raw`:

1. Select the columns `last_name`, `age`, `Drug`, `Dosage`, and `Diff`.
2. Filter participants older than 30 with last name `Durand`.
3. Filter participants who received drug `A` and dosage level 3.
4. Count how many rows are in each filtered result.

Self-check: the dataset has several `Durand` rows, so your second result should not be empty.


In [ ]:
# Write your solution here.
# Tip: build one condition at a time, then combine conditions with &.


# 4. Sorting data

Sorting helps you find high and low values, check unusual records, and prepare tables for reading.


In [ ]:
# Sort by age from largest to smallest.
people.sort_values("age", ascending=False)

In [ ]:
# Sort by city alphabetically, then by income from largest to smallest within each city.
people.sort_values(["city", "income"], ascending=[True, False])

You can also sort with a helper function. This example sorts names by their length.

The `lambda column: column.str.len()` part is a short, unnamed function. It tells pandas: "before sorting this text column, calculate the length of each value and sort by that length."

In [ ]:
# key=... transforms the column before sorting; here we sort names by text length.
people.sort_values("name", key=lambda column: column.str.len())

## Practice: Sorting

Using `islanders_raw`, show the 10 participants with the largest positive `Diff`. Then show the 10 participants with the most negative `Diff`.

What does a positive `Diff` mean in this dataset?


In [ ]:
# Write your solution here.


# 5. Creating new columns

New columns are often where analysis begins. You might create:

- a calculated value, such as a ratio or difference
- a category, such as age group
- a Boolean flag, such as whether a value is above a threshold

Prefer creating a new DataFrame or using `.assign()` while learning. It keeps your workflow easier to inspect.

In the next example:

- `.assign()` creates new columns and returns a new DataFrame
- `lambda df: ...` means "use the current DataFrame in this calculation"
- `pd.cut()` turns numeric values into labelled bins, such as age groups

In [ ]:
# .assign() creates new columns and returns a new DataFrame.
people_enriched = people.assign(
    # Calculate 24% of income.
    income_tax=lambda df: df["income"] * 0.24,
    # Calculate remaining income after that tax.
    income_after_tax=lambda df: df["income"] * 0.76,
    # Bin numeric ages into labelled groups.
    age_group=lambda df: pd.cut(
        df["age"],
        bins=[0, 24, 40, 120],
        labels=["under 25", "25 to 40", "over 40"],
    ),
    # Create a True/False flag for incomes at or above 1800.
    high_income=lambda df: df["income"] >= 1800,
)
people_enriched

In [ ]:
# Create a working copy for analysis columns.
islanders = islanders_raw.copy()

# Mark repeated labels as categorical variables.
islanders["Drug"] = islanders["Drug"].astype("category")
islanders["Happy_Sad_group"] = islanders["Happy_Sad_group"].astype("category")

# Convert numeric ages into age-group labels.
islanders["age_group"] = pd.cut(
    islanders["age"],
    bins=[0, 35, 50, 65, 120],
    labels=["25-35", "36-50", "51-65", "66+"],
)

# True means the after-treatment score is higher than the before-treatment score.
islanders["improved"] = islanders["Diff"] > 0

islanders.head()

# value_counts() is a quick way to check how many rows are in each category.
islanders["age_group"].value_counts()

## Practice: Create analysis variables

Using `islanders`:

1. Create `score_ratio` as `Mem_Score_After / Mem_Score_Before`.
2. Create `large_improvement` that is `True` when `Diff` is at least 10.
3. Create an age group column using your own age boundaries.
4. Check how many participants are in each new age group.


In [ ]:
# Write your solution here.
# Tip: .value_counts() is useful for checking categories.


# 6. Missing values

Missing values are common in real data. Do not automatically delete or fill them. First ask:

- Which columns contain missing values?
- How many rows are affected?
- Is the missingness likely to be random or related to a group?
- Will dropping or filling values change the question you are answering?

The example dataset has no obvious missing values, so we create a small messy copy to practise.


In [ ]:
# Start from a clean copy of the small example table.
messy_people = people.copy()

# Insert two missing values so we can practise detecting and handling them.
messy_people.loc[1, "income"] = pd.NA
messy_people.loc[3, "city"] = pd.NA

display(messy_people)
print("Missing values per column:")
display(messy_people.isna().sum())

In [ ]:
# Drop rows only when missingness makes the row unusable for your question.
without_missing_income = messy_people.dropna(subset=["income"])
display(without_missing_income)

# Fill values only when you can justify the replacement.
median_income = messy_people["income"].median()
filled_income = messy_people.assign(income=messy_people["income"].fillna(median_income))
filled_income


## Practice: Check missing values

Using `islanders`, count missing values per column. If there are no missing values, write one sentence explaining why you still checked.


In [ ]:
# Write your solution here.


## Concept test 1: pandas foundations

Take about 5 minutes. Try to answer without running code first, then use a small code cell to check anything you are unsure about.

## Multiple Choice Questions

1. **Which statement best describes a pandas `DataFrame`?**
   - A) A single Python value stored in memory.
   - B) A one-dimensional labelled column.
   - C) A two-dimensional table with rows and columns.
   - D) A plotting function for tabular data.

2. **Which expression selects rows where `Diff` is positive and keeps only `Drug` and `Diff`?**
   - A) `islanders["Diff" > 0, ["Drug", "Diff"]]`
   - B) `islanders.loc[islanders["Diff"] > 0, ["Drug", "Diff"]]`
   - C) `islanders.loc[["Drug", "Diff"], islanders["Diff"] > 0]`
   - D) `islanders.filter("Diff" > 0)`

## True/False Statements

3. **`islanders["Diff"] > 0` returns one True/False value for each row.**
   - True / False

4. **It is always safe to use `dropna()` before checking which values are missing.**
   - True / False

## Short Answer Question

5. **You create a new column called `age_group`. What should you check before using it in a grouped summary?**

<details>
<summary>Check your answers</summary>

#### Multiple Choice Questions

1. C) A two-dimensional table with rows and columns.
2. B) `islanders.loc[islanders["Diff"] > 0, ["Drug", "Diff"]]`

#### True/False Statements

3. True
4. False. Dropping rows before checking missingness can silently remove useful data or bias the analysis.

#### Short Answer Question

5. Check that each row received the expected category and that group sizes are reasonable, for example with `.value_counts()`.

</details>

# 7. Combining tables

`merge` joins two tables using a shared key column, similar to a database join. This is useful when one table contains measurements and another table contains metadata.


In [ ]:
# A lookup table maps short drug codes to readable labels and metadata.
drug_lookup = pd.DataFrame({
    "Drug": ["A", "T", "S"],
    "drug_name": ["Alprazolam", "Triazolam", "Placebo"],
    "drug_type": ["anti-anxiety", "anti-anxiety", "control"],
})

# Merge adds matching lookup columns to each row of the main data.
islanders_named = islanders.merge(drug_lookup, on="Drug", how="left")
islanders_named.head()

Common join types:

- `how="left"`: keep all rows from the left table and add matches from the right table
- `how="inner"`: keep only rows with matches in both tables
- `how="outer"`: keep all rows from both tables

Before merging, check that the key columns use compatible data types and contain the values you expect.


In [ ]:
# Check that both tables use the same drug codes before merging.
print("Drug codes in data:", sorted(islanders["Drug"].astype(str).unique()))
print("Drug codes in lookup:", sorted(drug_lookup["Drug"].unique()))

`concat` stacks tables by rows or places tables side by side by columns. Use it when tables already have the same structure or matching row order.


In [ ]:
# Split the table into two row blocks.
top_rows = people.iloc[:2]
bottom_rows = people.iloc[2:]

# Stack the blocks back together row-wise.
combined_rows = pd.concat([top_rows, bottom_rows], axis=0)
combined_rows

## Practice: Merge metadata

Create a small lookup table for `Happy_Sad_group` with labels `Happy memory priming` and `Sad memory priming`. Merge it into `islanders_named` and call the result `islanders_labeled`.


In [ ]:
# Write your solution here.


# 8. Grouping and aggregation

`groupby` splits data into groups, applies a calculation to each group, and combines the result.

Useful aggregation functions include:

- `mean`: average
- `median`: middle value
- `min`, `max`: smallest and largest
- `count`: number of non-missing values
- `size`: number of rows
- `std`: standard deviation

This pattern is central for exploratory data analysis and later statistical thinking.


In [ ]:
# Group rows by city and calculate named summary columns.
city_summary = people.groupby("city").agg(
    mean_age=("age", "mean"),
    total_income=("income", "sum"),
    n_people=("name", "size"),
)
city_summary

In [ ]:
# Group by both drug and dosage to compare combinations of categories.
drug_summary = islanders_named.groupby(["drug_name", "Dosage"]).agg(
    n=("Diff", "size"),
    mean_diff=("Diff", "mean"),
    median_diff=("Diff", "median"),
    min_diff=("Diff", "min"),
    max_diff=("Diff", "max"),
)

# reset_index() turns grouped index labels back into ordinary columns.
drug_summary = drug_summary.reset_index()

# Sort so related rows appear together.
drug_summary.sort_values(["drug_name", "Dosage"])

Optional extension: you can use your own aggregation function. Here we calculate a simple range: maximum minus minimum.

In [ ]:
# Define a custom summary function.
def value_range(series):
    return series.max() - series.min()

# Use the custom function inside .agg().
islanders_named.groupby("drug_name").agg(
    diff_range=("Diff", value_range),
    mean_age=("age", "mean"),
)

## Practice: Group and interpret

Using `islanders_named`:

1. Group by `drug_name` and calculate the mean, median, and standard deviation of `Diff`.
2. Group by `drug_name` and `age_group` and calculate the mean `Diff` and number of rows.
3. Which group has the highest mean improvement? How many participants are in that group?

Self-study note: a group mean from a small group can be unstable. Always check the group size.


In [ ]:
# Write your solution here.


# 9. Pivot tables and reshaping

A pivot table summarises values into a matrix. It is useful for comparing combinations of categories.

In the next example:

- `values="Diff"` is the numeric column to summarise
- `index="drug_name"` gives the row groups
- `columns="Dosage"` gives the column groups
- `aggfunc="mean"` says to calculate the mean for each group combination

In [ ]:
# Summarise mean Diff for each drug-by-dosage combination.
pivot_diff = pd.pivot_table(
    islanders_named,
    values="Diff",       # numeric column to summarise
    index="drug_name",   # row groups
    columns="Dosage",    # column groups
    aggfunc="mean",      # summary statistic
)
pivot_diff

Use `melt` to reshape wide data into long format.

Wide data stores related measurements in separate columns, such as `Mem_Score_Before` and `Mem_Score_After`. Long data stores those measurements in one value column and uses another column to say what kind of measurement each row contains.

In the next example:

- `id_vars` are columns that identify or describe the participant and should stay as they are
- `value_vars` are the measurement columns to stack into one column
- `var_name` is the name of the new column that stores the old measurement column names
- `value_name` is the name of the new column that stores the measurement values

Long format is useful for later grouping, comparison, visualisation, and statistical modelling because each row has a consistent meaning.

In [ ]:
# Reshape before/after memory scores into one long value column.
score_long = islanders_named.melt(
    id_vars=["first_name", "last_name", "age", "age_group", "Drug", "drug_name", "Dosage", "Happy_Sad_group"],
    value_vars=["Mem_Score_Before", "Mem_Score_After"],
    var_name="timepoint",
    value_name="memory_score",
)

score_long.head()

In [ ]:
# Once data is long, group by the measurement type just like any other category.
score_summary = score_long.groupby(["drug_name", "timepoint"]).agg(
    mean_score=("memory_score", "mean"),
    n=("memory_score", "size"),
)

# reset_index() turns grouped index labels back into ordinary columns.
score_summary.reset_index().head(10)

## Practice: Pivot and reshape

1. Create a pivot table showing mean `Mem_Score_After` by `Drug` and `Dosage`.
2. Use `melt` to create a long table with `Mem_Score_Before`, `Mem_Score_After`, and `Diff` in one value column.
3. Explain why long format is useful before later plotting or statistical analysis.

In [ ]:
# Write your solution here.


# 10. Optional: A readable workflow with `pipe`

When analysis has several steps, `pipe` lets you name each step and chain them together. This is optional for this course. If you have used R and the tidyverse, the idea may feel familiar.

Use `pipe` when it makes your workflow easier to read. It is not required for the exercises.

In [ ]:
# Each helper function performs one named data-preparation step.
def add_age_group(df):
    return df.assign(
        age_group=pd.cut(
            df["age"],
            bins=[0, 35, 50, 65, 120],
            labels=["25-35", "36-50", "51-65", "66+"],
        )
    )


def add_drug_labels(df):
    return df.merge(drug_lookup, on="Drug", how="left")


def keep_complete_scores(df):
    return df.dropna(subset=["Mem_Score_Before", "Mem_Score_After", "Diff"])

# pipe() applies each helper function in order.
analysis_df = (
    islanders_raw
    .pipe(keep_complete_scores)
    .pipe(add_age_group)
    .pipe(add_drug_labels)
    .assign(improved=lambda df: df["Diff"] > 0)
)

analysis_df.head()

# Independent task: short pandas checkpoint

Take about 10 minutes. Work independently or in pairs.

Use the `Islander_data` dataset that is already loaded in this notebook. Your goal is to make one small summary table and write a short interpretation.

Tasks:

1. Make a copy of the dataset for your work.
2. Check how many rows and columns it has.
3. Check how many missing values are in `Diff`.
4. Keep rows where `Diff` is not missing.
5. Group by `Drug` and calculate:
   - mean `Diff`
   - number of participants
6. Write one or two sentences: which drug group has the highest average `Diff`, and why should we also look at the participant count?

Optional extension, if you finish early: repeat the summary by both `Drug` and `Dosage`.

In [ ]:
# Start your short pandas checkpoint here.
# Suggested first step:
# analysis = islanders_raw.copy()


## Final concept test: preparing data for analysis

Take about 5 minutes. These questions connect the pandas skills to later statistical work.

## Multiple Choice Questions

1. **When should you use `merge()` rather than `concat()`?**
   - A) When joining tables by a shared key column.
   - B) When sorting one table by a numeric column.
   - C) When checking missing values.
   - D) When renaming columns.

2. **Why is it useful to calculate `n` together with a group mean?**
   - A) `n` converts the mean into a percentage.
   - B) `n` shows how many observations support the mean.
   - C) `n` removes missing values from the original table.
   - D) `n` sorts the result alphabetically.

## True/False Statements

3. **`pivot_table()` can aggregate repeated row and column combinations.**
   - True / False

4. **Long format can make repeated measurements easier to compare because the measurement type and value are stored in consistent columns.**
   - True / False

## Short Answer Question

5. **Look at this code: `df.groupby("Drug").agg(mean_diff=("Diff", "mean"))`. What does each output row represent?**

<details>
<summary>Check your answers</summary>

#### Multiple Choice Questions

1. A) When joining tables by a shared key column.
2. B) `n` shows how many observations support the mean.

#### True/False Statements

3. True
4. True

#### Short Answer Question

5. Each output row represents one drug group and its mean `Diff` value.

</details>

## What you should feel ready for next

After this notebook, the next statistics course will be easier if you can comfortably:

- read a CSV file into a DataFrame
- select and filter rows with `.loc`
- create analysis variables
- group data and calculate summary statistics
- reshape data to long format for plotting
- explain what each row and column in your analysis table represents

If any of those still feel uncertain, revisit the matching section and redo the practice task with a small change.


## Further learning

- [Pandas Getting started tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html)
- [Pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- [Series reference](https://pandas.pydata.org/docs/reference/series.html)
- [DataFrame reference](https://pandas.pydata.org/docs/reference/frame.html)

When using documentation, check the pandas version. This matters because pandas functions can change over time: a method may gain or lose parameters, defaults may change, and examples written for a newer version may not run the same way in an older one. In Python, run `pd.__version__` to see the version you are using.
